# EDA notebook

Quick look at the ETF volatility panel: coverage, gaps, distributions, simple autocorrelation, train-only feature/target checks, a few leakage spot checks, and simple persistence baselines.

In [ ]:
import os, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

try:
    import seaborn as sns
    HAS_SNS = True
except ImportError:
    HAS_SNS = False

try:
    from statsmodels.tsa.stattools import acf
    HAS_SM = True
except ImportError:
    HAS_SM = False

print('seaborn:', HAS_SNS, '| statsmodels:', HAS_SM)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_PATH  = '/content/drive/MyDrive/Colab Notebooks/vol_dataset_post_wrangle_021026.csv'
OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

DATE_COL   = 'date'
TICKER_COL = 'ticker'
PRICE_COL  = 'adj_close'
VOL_COL    = 'volume'
RET_COL    = 'daily_log_return'
TARGET_COL = 'forward_vol_5d_annual_decimel_calculated'

TRAIL_VOL_COLS = [
    'trailing_vol_annual_decimel_5d_calculated',
    'trailing_vol_annual_decimel_20d_calculated',
    'trailing_vol_annual_decimel_5d_factset',
    'trailing_vol_annual_decimel_20d_factset',
]
MACRO_COLS = ['US_3M_TB_YLD', 'US_10Y_BOND_YLD', 'NYGOLDS', 'OIL_WTI_S', 'VIX']

SAMPLE_TICKER = 'SPY'
BIG_GAP_DAYS  = 10
ANNUALIZATION = 252
TRAIN_END = '2022-12-31'
VAL_END   = '2024-12-31'

In [ ]:
df_raw = pd.read_csv(DATA_PATH, low_memory=False)

df = df_raw.copy()
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')

assert df[DATE_COL].isna().sum() == 0, 'some dates failed to parse'

df = df.sort_values([TICKER_COL, DATE_COL]).reset_index(drop=True)

print(df.shape, '|', df[TICKER_COL].nunique(), 'tickers')
print(df[DATE_COL].min().date(), '->', df[DATE_COL].max().date())
df.head()

## Panel checks

In [ ]:
# duplicate rows
print('duplicate (ticker, date) rows:', df.duplicated([DATE_COL, TICKER_COL]).sum())

# dates in order within ticker
non_mono = (~df.groupby(TICKER_COL)[DATE_COL].apply(lambda s: s.is_monotonic_increasing)).sum()
print('tickers with non-monotone dates:', non_mono)

# basic coverage by ticker
coverage = (
    df.groupby(TICKER_COL)[DATE_COL]
      .agg(start='min', end='max', n_rows='count')
      .sort_values('n_rows', ascending=False)
)
print('tickers:', coverage.shape[0])
display(coverage.head(10))
display(coverage.tail(5))

In [ ]:
# look for big calendar gaps
tmp = df[[TICKER_COL, DATE_COL]].copy()
tmp['prev'] = tmp.groupby(TICKER_COL)[DATE_COL].shift(1)
tmp['gap']  = (tmp[DATE_COL] - tmp['prev']).dt.days
big_gaps = tmp[tmp['gap'] > BIG_GAP_DAYS]
print(f'big gaps (>{BIG_GAP_DAYS}d):', len(big_gaps))
display(big_gaps.head(20))

# missingness by column
miss_tbl = pd.DataFrame({
    'missing_rate':  df.isna().mean(),
    'missing_count': df.isna().sum()
}).sort_values('missing_rate', ascending=False)
display(miss_tbl.head(15))

# target missingness by ticker
if TARGET_COL in df.columns:
    display(df.groupby(TICKER_COL)[TARGET_COL]
              .apply(lambda s: s.isna().mean())
              .sort_values(ascending=False).head(10))

In [ ]:
# macro columns should match across tickers on the same date
present_macro = [c for c in MACRO_COLS if c in df.columns]

macro_check = pd.DataFrame({
    c: {
        'median_unique': float(df.groupby(DATE_COL)[c].nunique(dropna=False).median()),
        'max_unique':    int(df.groupby(DATE_COL)[c].nunique(dropna=False).max()),
        'pct_dates_gt1': float((df.groupby(DATE_COL)[c].nunique(dropna=False) > 1).mean()),
    }
    for c in present_macro
}).T
display(macro_check)

## Distributions

In [ ]:
def hist(series, title, bins=100):
    s = series.dropna()
    plt.figure(figsize=(7, 3))
    plt.hist(s, bins=bins)
    plt.title(title)
    plt.show()

if RET_COL in df.columns:
    hist(df[RET_COL], 'daily log returns (pooled)')
    print(df[RET_COL].describe(percentiles=[.01, .05, .5, .95, .99]))

if VOL_COL in df.columns:
    hist(np.log(df[VOL_COL].replace(0, np.nan)), 'log(volume) (pooled)')

for c in [x for x in TRAIL_VOL_COLS if x in df.columns]:
    hist(df[c], c)

if TARGET_COL in df.columns:
    hist(df[TARGET_COL], f'target: {TARGET_COL}')
    print(df[TARGET_COL].describe(percentiles=[.01, .05, .5, .95, .99]))

## SPY checks

In [ ]:
d = df[df[TICKER_COL] == SAMPLE_TICKER].copy()
print(SAMPLE_TICKER, d.shape[0], 'rows |', d[DATE_COL].min().date(), '->', d[DATE_COL].max().date())

for col, title in [
    (PRICE_COL,  'Adjusted close'),
    (RET_COL,    'Daily log returns'),
    (TARGET_COL, 'Target: forward 5d vol'),
]:
    if col in d.columns:
        plt.figure(figsize=(10, 3))
        plt.plot(d[DATE_COL], d[col])
        plt.title(f'{SAMPLE_TICKER} — {title}')
        plt.show()

for c in [x for x in TRAIL_VOL_COLS if x in d.columns]:
    plt.figure(figsize=(10, 3))
    plt.plot(d[DATE_COL], d[c])
    plt.title(f'{SAMPLE_TICKER} — {c}')
    plt.show()

for c in [x for x in MACRO_COLS if x in d.columns]:
    plt.figure(figsize=(10, 3))
    plt.plot(d[DATE_COL], d[c])
    plt.title(f'macro: {c}')
    plt.show()

## Autocorrelation

In [ ]:
d = df[df[TICKER_COL] == SAMPLE_TICKER].copy()

def plot_acf(x, nlags=30, title='ACF'):
    if HAS_SM:
        vals = acf(x, nlags=nlags, fft=True)
    else:
        vals = np.array([1.0] + [np.corrcoef(x[:-k], x[k:])[0,1] for k in range(1, nlags+1)])
    plt.figure(figsize=(7, 3))
    plt.stem(range(len(vals)), vals)
    plt.title(title)
    plt.xlabel('lag')
    plt.show()

if RET_COL in d.columns:
    r  = d[RET_COL].dropna().values
    r2 = r ** 2
    plot_acf(r,  title=f'{SAMPLE_TICKER} ACF — returns')
    plot_acf(r2, title=f'{SAMPLE_TICKER} ACF — squared returns')

if TARGET_COL in d.columns:
    plot_acf(d[TARGET_COL].dropna().values, title=f'{SAMPLE_TICKER} ACF — target forward vol')

## Train-only feature vs target

In [ ]:
df_train = df[df[DATE_COL] <= pd.to_datetime(TRAIN_END)].copy()
print('train rows:', len(df_train), '| through', TRAIN_END)

feat_cols = [c for c in TRAIL_VOL_COLS + MACRO_COLS if c in df_train.columns and c != TARGET_COL]

if TARGET_COL in df_train.columns:
    corr = df_train[feat_cols + [TARGET_COL]].corr(numeric_only=True)[[TARGET_COL]].sort_values(TARGET_COL, ascending=False)
    display(corr)

    for c in corr.index[1:5]:
        plt.figure(figsize=(5, 4))
        plt.scatter(df_train[c], df_train[TARGET_COL], s=3, alpha=0.3)
        plt.xlabel(c)
        plt.ylabel(TARGET_COL)
        plt.title(f'{c} vs target (train)')
        plt.show()

## Leakage spot checks

Recompute a few pieces from price data and compare to the provided columns.

In [ ]:
d = df[df[TICKER_COL] == SAMPLE_TICKER].copy().sort_values(DATE_COL)

# recompute log returns
if PRICE_COL in d.columns:
    d['ret_recomp'] = np.log(d[PRICE_COL] / d[PRICE_COL].shift(1))
    if RET_COL in d.columns:
        diff = (d[RET_COL] - d['ret_recomp']).abs()
        print('return diff — mean:', round(diff.mean(), 6), '| median:', round(diff.median(), 6))
        plt.figure(figsize=(10, 3))
        plt.plot(d[DATE_COL], diff)
        plt.title(f'{SAMPLE_TICKER} — |provided return - recomputed|')
        plt.show()

# recompute trailing vol
def trail_vol(ret, window):
    return ret.rolling(window).std(ddof=1) * math.sqrt(ANNUALIZATION)

if 'ret_recomp' in d.columns:
    d['tv5_recomp']  = trail_vol(d['ret_recomp'], 5)
    d['tv20_recomp'] = trail_vol(d['ret_recomp'], 20)

    for col, recomp in [
        ('trailing_vol_annual_decimel_5d_calculated',  'tv5_recomp'),
        ('trailing_vol_annual_decimel_20d_calculated', 'tv20_recomp'),
    ]:
        if col not in d.columns:
            continue
        valid = d[[DATE_COL, col, recomp]].dropna()
        if len(valid) > 50:
            print(f'trail vol corr ({col}):', round(valid[col].corr(valid[recomp]), 4))
            plt.figure(figsize=(10, 3))
            plt.plot(valid[DATE_COL], valid[col],    label='provided')
            plt.plot(valid[DATE_COL], valid[recomp], label='recomputed', alpha=0.7)
            plt.title(f'{SAMPLE_TICKER} — {col}')
            plt.legend()
            plt.show()

# recompute forward vol label
if 'ret_recomp' in d.columns:
    d['fwd_vol_recomp'] = d['ret_recomp'].shift(-1).rolling(5).std(ddof=1) * math.sqrt(ANNUALIZATION)
    if TARGET_COL in d.columns:
        valid = d[[DATE_COL, TARGET_COL, 'fwd_vol_recomp']].dropna()
        if len(valid) > 50:
            print('fwd vol corr:', round(valid[TARGET_COL].corr(valid['fwd_vol_recomp']), 4))
            plt.figure(figsize=(10, 3))
            plt.plot(valid[DATE_COL], valid[TARGET_COL],    label='provided')
            plt.plot(valid[DATE_COL], valid['fwd_vol_recomp'], label='recomputed', alpha=0.7)
            plt.title(f'{SAMPLE_TICKER} — target vs recomputed forward vol')
            plt.legend()
            plt.show()

## Persistence baselines

In [ ]:
assert TARGET_COL in df.columns

train_mask = df[DATE_COL] <= pd.to_datetime(TRAIN_END)
val_mask   = (df[DATE_COL] > pd.to_datetime(TRAIN_END)) & (df[DATE_COL] <= pd.to_datetime(VAL_END))
test_mask  = df[DATE_COL] > pd.to_datetime(VAL_END)

def rmse(yt, yp): return float(np.sqrt(((yt - yp).dropna()**2).mean()))
def mae(yt, yp):  return float((yt - yp).dropna().abs().mean())

candidates = [c for c in [
    'trailing_vol_annual_decimel_5d_calculated',
    'trailing_vol_annual_decimel_20d_calculated'
] if c in df.columns]

rows = []
for feat in candidates:
    for split, mask in [('train', train_mask), ('val', val_mask), ('test', test_mask)]:
        sub = df.loc[mask, [TARGET_COL, feat]].dropna()
        if len(sub) == 0: continue
        rows.append({'feature': feat, 'split': split, 'n': len(sub),
                     'RMSE': rmse(sub[TARGET_COL], sub[feat]),
                     'MAE':  mae(sub[TARGET_COL],  sub[feat])})

display(pd.DataFrame(rows).sort_values(['feature', 'split']))

In [ ]:
eda_dir = 'eda_outputs'
os.makedirs(eda_dir, exist_ok=True)

coverage.to_csv(os.path.join(eda_dir, 'coverage.csv'))
miss_tbl.to_csv(os.path.join(eda_dir, 'missingness.csv'))
macro_check.to_csv(os.path.join(eda_dir, 'macro_check.csv'))

print('saved to', eda_dir)